# long generations num_generation ablation

some back of the envelope calcs for cost of these experiments

* old experiments (200 samples, 10 generations)
    * entailment predictions 2546
    * of those hashed 855 
    * total gpt-4 queries: 2546 - 855 = 1691
    * number of datapoints: 200
    * (queries for 400 datapoints: 3382)
    * total cost: 10
    * cost per prediction: 10/1691


* new epxeriments interim (400 samples (running still), 15 generations)
    * entailment predictions: 1179
    * of those hashed: 704
    * total gpt-4 queries:  1179 - 704 = 475
    * number of datapoints so far: 28
    * expected gpt-4 queries over 400 examples: 6785
    * expected total cost: 10/1691*6785 = 40


* cost on squad (still running, 15 generations)
    * entailment predictions: 29394
    * of those hashed: 5216
    * total gpt-4 queries: 24178
    * number of datapoints so far: 294
    * expected gpt-4 queries for 400 examples: 32895
    * expected total cost: 10/1691 * 32895 = 200



In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('..')
import pickle
import yaml
import json

from matplotlib import pyplot as plt
import pandas as pd
import numpy as np


from copy import deepcopy
import seaborn as sns

from IPython.display import display

import wandb
api = wandb.Api()
api.entity = 'goatml'

In [2]:
# export to eval file
def get_vals(x):
    return x['mean'], x['mean']-x['bootstrap']['std_err'], x['mean']+x['bootstrap']['std_err']
    # return x['mean'], x['bootstrap']['low'], x['bootstrap']['high']

def get_vals2(x):
    return x['means'], x['low'], x['high']


def get_perfs(metrics):
    data = []
    for method in metrics['performance']:
        mean, low, high = get_vals(metrics['performance'][method])
        data.append([method, mean, low, high])
    
    df = pd.DataFrame(data, columns=['method', 'means', 'low', 'high'])
    return df.set_index('method')

def plot_bar(df, color='C0', ax=None):
    if ax is None:
        fig, ax = plt.subplots()

    df.plot.bar(y='means', yerr=[df.means - df.low, df.high - df.means], color=color, ax=ax)

def get_uncertainty_df(metrics):
    data = []
    for method in metrics['uncertainty']:
        for metric in metrics['uncertainty'][method]:
            mean, low, high = get_vals(metrics['uncertainty'][method][metric])
            data.append([method, metric, mean, low, high])
    
    df = pd.DataFrame(data, columns=['method', 'metric', 'means', 'low', 'high'])
    
    df = df.set_index('method')

    def get_base_method(x):
        if 'p_false' in x:
            return 'p_false'
        elif 'p_ik' in x:
            return 'p_ik'
        elif 'regular_entropy' in x:
            return 'regular_entropy'
        elif 'cluster_assignment' in x:
            return x
        elif 'semantic_entropy' in x:
            return 'semantic_entropy'
        else:
            raise
    df['base_method'] = df.index.map(get_base_method)
    return df

def plot_bar_uncertainty(pdf, metric, remove='_UNANSWERABLE', ax=None):
    if ax is None:
        fig, ax = plt.subplots()

    tmp = pdf
    # filter out unanswerable
    if remove is not None:
        tmp = tmp[tmp.index.map(lambda x: remove not in x)]
    tmp = tmp.reset_index()
    tmp = tmp.set_index('metric').loc[metric]
    tmp = tmp.set_index('method')
    tmp = tmp.sort_values('base_method')

    b2c = {m:f'C{i}' for i, m in enumerate(tmp.base_method.unique())}
    basemethod2color = lambda x: b2c[x]

    plot_bar(tmp, color=tmp.base_method.map(basemethod2color), ax=ax)
    ax.grid(axis='y', zorder=-10, alpha=0.3)

In [3]:
def restore_file(wandb_id, filenames=['wandb-summary.json', 'config.yaml']):
    files_dir = 'notebooks/restored_files'    
    os.system(f'mkdir -p {files_dir}')

    run = api.run(f'goatml/semantic_uncertainty/{wandb_id}')
    # run = api.run(f'goatml/uncertainty/{wandb_id}')
    out = []

    for filename in filenames:
        
        path = f'{files_dir}/{filename}'
        os.system(f'rm -rf {path}')
        run.file(filename).download(root=files_dir, replace=True, exist_ok=False)
    
        if filename.endswith('.pkl'):
            with open(path, 'rb') as f:
                out.append(pickle.load(f))
        elif filename.endswith('.yaml'):
            with open(path, 'r') as f:
                out.append(yaml.safe_load(f))
        elif filename.endswith('.json'):
            with open(path, 'r') as f:
                out.append(json.load(f))
        else:
            raise

    return out

In [83]:
runs = {
    # '1gxo9oef': ['trivia_qa-llama2', ['config.yaml', 'validation_generations.pkl']],
    # 'dqtye228': ['bioasq-llama2', ['config.yaml', 'validation_generations.pkl']],
    'uvfbxm6d': ['squad-llama2', ['config.yaml', 'validation_generations.pkl']],
    # we stored updated accuracies in uncertainty measures only
    # 'lgurwdvq': ['trivia_qa-gpt35', ['config.yaml', 'uncertainty_measures.pkl']],
    # 'nu83s9tf': ['bioasq-gpt35', ['config.yaml', 'uncertainty_measures.pkl']]
    'm3y3x605': ['squad-gpt35', ['config.yaml', 'uncertainty_measures.pkl']],
}

In [84]:
all_results, uncertainty = {}, {}

for wandb_id, (name, files) in runs.items():
    all_configs[wandb_id], all_results[wandb_id] = restore_file(wandb_id, filenames=files)


In [68]:
# For bioasq there are (16/200, 0.08) datapoints where the classifiers disagree --> on those I have manually judged gpt 35 to be better on 15 (and unclear on one)
# For trivia_qa there are (14/400, 0.035) disagreements --> of those I have judged 4 where gpt is better 5 where llama is better and 5 where the label is bad
# For squad there are (48-> 400, 0.12) disagreements --> I have looked until including idx 188 (i.e. half the disagreements and I have found) 1 instance of a bad question, 12 instances where gpt is better, and 10 instances where llama is better


# Qualitative difference: llama is a bit more literal (i.e. chopin = 'famous composer and pianist' does not count if label was musician. also llama is worse at detecting model answer with nuance to a yes or no question)
# For gpt it looks a little bit like it sometimes ignores the true answer! i.e. if the model answers something true, but it does not match the label, gpt will count it as true


16

In [85]:
llama2id = 'uvfbxm6d'
gpt35id = 'm3y3x605'

In [86]:
disagreements = []
for i, key in enumerate(all_results[llama2id]):
    llama2 = all_results[llama2id][key]

    gpt_accuracy = 1 - all_results[gpt35id]['validation_is_false'][i]

    disagreement = llama2['most_likely_answer']['accuracy'] != gpt_accuracy
    disagreements.append(disagreement)

    if disagreement:
        print(f'XXXX DISAGREEMENT {len(disagreements)} XXXX')
        print('Q:', llama2['question'])
        print('True answer:', llama2['reference']['answers']['text'])
        print('Model response:', llama2['most_likely_answer']['response'])
        print('llama2 acc ', llama2['most_likely_answer']['accuracy'], 'gpt35 acc', gpt_accuracy)


XXXX DISAGREEMENT 3 XXXX
Q: Who was Frédéric Chopin?
True answer: ['Famous musicians', 'musicians', 'Famous musicians']
Model response: Frédéric Chopin was a Polish composer and pianist of the Romantic era, known for his expressive and technically demanding piano music, particularly his nocturnes, waltzes, and preludes.
llama2 acc  0.0 gpt35 acc 1.0
XXXX DISAGREEMENT 5 XXXX
Q: Where did Moncalm slip away to attack, left largely unprotected?
True answer: ['Oswego', 'Oswego', 'Oswego', 'Oswego', 'Oswego']
Model response: Moncalm slipped away to attack the British left flank, which was left largely unprotected.
llama2 acc  0.0 gpt35 acc 1.0
XXXX DISAGREEMENT 12 XXXX
Q: After 1935, who would be forbidden to enter Rhineland?
True answer: ['German army', 'German army', 'German army']
Model response: After 1935, military forces from any country except Germany would be forbidden to enter Rhineland under the terms of the Treaty of Versailles and the subsequent Rhineland Pact.
llama2 acc  0.0 gp

In [87]:
np.sum(disagreements), len(disagreements), np.mean(disagreements)

(48, 400, 0.12)